# 02 - EfficientNet-B0 Training & Evaluation

**Phase 2 of PCB Surface Defect Inspection System**

This notebook covers:
- Task 2.1: Model setup (EfficientNet-B0 + grayscale adaptation)
- Task 2.2: Training loop (mixed precision, AdamW, cosine LR)
- Task 2.3: Hyperparameter tuning
- Task 2.4: Evaluation (confusion matrix, per-class F1)
- Task 2.5: Grad-CAM visualization

**Runtime:** Colab T4/A100 GPU  
**Estimated time:** 20-30 minutes

## Setup

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Install dependencies
!pip install -q albumentations opencv-python-headless mlflow statsmodels onnx onnxruntime

In [ ]:
# Project paths (all on Google Drive)
import os
import sys

# === UPDATE THIS PATH TO YOUR DRIVE FOLDER ===
DRIVE_ROOT = "/content/drive/MyDrive/defect_inspection"

# Paths (matches your Drive structure: data/raw/NEU-DET)
DATA_DIR = f"{DRIVE_ROOT}/data/raw/NEU-DET"
MODELS_DIR = f"{DRIVE_ROOT}/models"
RESULTS_DIR = f"{DRIVE_ROOT}/results"
SRC_DIR = f"{DRIVE_ROOT}/src"

# Add src to Python path
sys.path.insert(0, DRIVE_ROOT)

# Create output directories
os.makedirs(MODELS_DIR, exist_ok=True)
os.makedirs(f"{RESULTS_DIR}/gradcam", exist_ok=True)
os.makedirs(f"{RESULTS_DIR}/logs", exist_ok=True)

# Verify structure
print("Checking project structure...")
print(f"  Data:    {os.path.exists(DATA_DIR)} - {DATA_DIR}")
print(f"  Src:     {os.path.exists(SRC_DIR)} - {SRC_DIR}")
print(f"  Models:  {os.path.exists(MODELS_DIR)} - {MODELS_DIR}")

if not os.path.exists(DATA_DIR):
    print("\n⚠️  Data directory not found! Upload NEU-DET to:")
    print(f"   {DATA_DIR}")
if not os.path.exists(SRC_DIR):
    print("\n⚠️  Source code not found! Upload 'src' folder to:")
    print(f"   {SRC_DIR}")

In [ ]:
# Verify GPU
import torch

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
    DEVICE = torch.device("cuda")
else:
    print("WARNING: No GPU detected! Training will be slow.")
    DEVICE = torch.device("cpu")

print(f"\nUsing device: {DEVICE}")

## Task 2.1 — Model Setup

In [ ]:
import torch
import torch.nn as nn
from torchvision import models


def create_model(num_classes=6, pretrained=True):
    """
    Create EfficientNet-B0 with grayscale adaptation.
    
    The pretrained weights expect 3-channel RGB input.
    For grayscale, we average the first conv weights across channels.
    """
    # Load pretrained EfficientNet-B0
    if pretrained:
        weights = models.EfficientNet_B0_Weights.IMAGENET1K_V1
        model = models.efficientnet_b0(weights=weights)
    else:
        model = models.efficientnet_b0(weights=None)
    
    # Grayscale adaptation: average RGB weights for first conv
    # Original: (32, 3, 3, 3) -> New: (32, 1, 3, 3) expanded to (32, 3, 3, 3)
    # We keep 3 channels but use averaged weights so grayscale input works
    first_conv = model.features[0][0]
    with torch.no_grad():
        # Average across input channels
        avg_weight = first_conv.weight.mean(dim=1, keepdim=True)
        # Replicate to 3 channels (our preprocessing outputs 3-channel grayscale)
        first_conv.weight = nn.Parameter(avg_weight.repeat(1, 3, 1, 1))
    
    # Replace classifier head
    # EfficientNet-B0 has 1280 features before final fc
    model.classifier = nn.Sequential(
        nn.Dropout(p=0.2, inplace=True),
        nn.Linear(1280, num_classes)
    )
    
    return model


# Test model creation
model = create_model(num_classes=6)
model = model.to(DEVICE)

# Verify forward pass
dummy_input = torch.randn(4, 3, 224, 224).to(DEVICE)
with torch.no_grad():
    output = model(dummy_input)

print(f"Model created successfully")
print(f"Input shape: {dummy_input.shape}")
print(f"Output shape: {output.shape}")
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")

## Load Dataset

In [ ]:
from src.dataset import get_dataloaders, CLASS_NAMES
from src.augmentations import get_train_transforms, get_val_transforms

# Create dataloaders
train_loader, val_loader, test_loader = get_dataloaders(
    root=DATA_DIR,
    batch_size=32,
    num_workers=2,
    train_transform=get_train_transforms(),
    val_transform=get_val_transforms(),
    preprocess=True,
    seed=42
)

print(f"\nDataloaders created:")
print(f"  Train batches: {len(train_loader)}")
print(f"  Val batches: {len(val_loader)}")
print(f"  Test batches: {len(test_loader)}")

## Task 2.2 — Training Loop

In [ ]:
import time
import csv
from tqdm.auto import tqdm
from sklearn.metrics import f1_score, accuracy_score


class Trainer:
    """
    Trainer with mixed precision, early stopping, and checkpointing.
    """
    
    def __init__(
        self,
        model,
        train_loader,
        val_loader,
        device,
        lr=1e-3,
        weight_decay=1e-4,
        label_smoothing=0.1,
        epochs=50,
        patience=10,
        checkpoint_dir=MODELS_DIR,
        log_dir=f"{RESULTS_DIR}/logs"
    ):
        self.model = model
        self.train_loader = train_loader
        self.val_loader = val_loader
        self.device = device
        self.epochs = epochs
        self.patience = patience
        self.checkpoint_dir = checkpoint_dir
        self.log_dir = log_dir
        
        # Loss with label smoothing
        self.criterion = nn.CrossEntropyLoss(label_smoothing=label_smoothing)
        
        # Optimizer
        self.optimizer = torch.optim.AdamW(
            model.parameters(),
            lr=lr,
            weight_decay=weight_decay
        )
        
        # Cosine annealing scheduler
        self.scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
            self.optimizer,
            T_max=epochs,
            eta_min=1e-6
        )
        
        # Mixed precision
        self.scaler = torch.cuda.amp.GradScaler(enabled=(device.type == "cuda"))
        
        # Tracking
        self.best_val_f1 = 0.0
        self.patience_counter = 0
        self.history = []
    
    def train_epoch(self):
        self.model.train()
        total_loss = 0
        all_preds = []
        all_labels = []
        
        pbar = tqdm(self.train_loader, desc="Training")
        for images, labels in pbar:
            images = images.to(self.device)
            labels = labels.to(self.device)
            
            self.optimizer.zero_grad()
            
            # Mixed precision forward
            with torch.cuda.amp.autocast(enabled=(self.device.type == "cuda")):
                outputs = self.model(images)
                loss = self.criterion(outputs, labels)
            
            # Backward with gradient scaling
            self.scaler.scale(loss).backward()
            self.scaler.step(self.optimizer)
            self.scaler.update()
            
            total_loss += loss.item()
            preds = outputs.argmax(dim=1).cpu().numpy()
            all_preds.extend(preds)
            all_labels.extend(labels.cpu().numpy())
            
            pbar.set_postfix({"loss": f"{loss.item():.4f}"})
        
        avg_loss = total_loss / len(self.train_loader)
        accuracy = accuracy_score(all_labels, all_preds)
        f1 = f1_score(all_labels, all_preds, average="macro")
        
        return avg_loss, accuracy, f1
    
    @torch.no_grad()
    def validate(self):
        self.model.eval()
        total_loss = 0
        all_preds = []
        all_labels = []
        
        for images, labels in tqdm(self.val_loader, desc="Validation"):
            images = images.to(self.device)
            labels = labels.to(self.device)
            
            with torch.cuda.amp.autocast(enabled=(self.device.type == "cuda")):
                outputs = self.model(images)
                loss = self.criterion(outputs, labels)
            
            total_loss += loss.item()
            preds = outputs.argmax(dim=1).cpu().numpy()
            all_preds.extend(preds)
            all_labels.extend(labels.cpu().numpy())
        
        avg_loss = total_loss / len(self.val_loader)
        accuracy = accuracy_score(all_labels, all_preds)
        f1 = f1_score(all_labels, all_preds, average="macro")
        
        return avg_loss, accuracy, f1
    
    def save_checkpoint(self, epoch, val_f1, is_best=False):
        checkpoint = {
            "epoch": epoch,
            "model_state_dict": self.model.state_dict(),
            "optimizer_state_dict": self.optimizer.state_dict(),
            "scheduler_state_dict": self.scheduler.state_dict(),
            "val_f1": val_f1,
            "best_val_f1": self.best_val_f1,
        }
        
        # Save latest
        torch.save(checkpoint, f"{self.checkpoint_dir}/latest_checkpoint.pth")
        
        # Save best
        if is_best:
            torch.save(checkpoint, f"{self.checkpoint_dir}/best_model.pth")
            print(f"  Saved best model (F1: {val_f1:.4f})")
    
    def save_history(self):
        with open(f"{self.log_dir}/training_log.csv", "w", newline="") as f:
            writer = csv.DictWriter(f, fieldnames=self.history[0].keys())
            writer.writeheader()
            writer.writerows(self.history)
    
    def train(self):
        print(f"\nStarting training for {self.epochs} epochs...")
        print(f"Device: {self.device}")
        print(f"Checkpoints: {self.checkpoint_dir}")
        print()
        
        start_time = time.time()
        
        for epoch in range(1, self.epochs + 1):
            print(f"Epoch {epoch}/{self.epochs}")
            print("-" * 40)
            
            # Train
            train_loss, train_acc, train_f1 = self.train_epoch()
            
            # Validate
            val_loss, val_acc, val_f1 = self.validate()
            
            # Update scheduler
            self.scheduler.step()
            current_lr = self.scheduler.get_last_lr()[0]
            
            # Log
            self.history.append({
                "epoch": epoch,
                "train_loss": train_loss,
                "train_acc": train_acc,
                "train_f1": train_f1,
                "val_loss": val_loss,
                "val_acc": val_acc,
                "val_f1": val_f1,
                "lr": current_lr
            })
            
            print(f"  Train - Loss: {train_loss:.4f}, Acc: {train_acc:.4f}, F1: {train_f1:.4f}")
            print(f"  Val   - Loss: {val_loss:.4f}, Acc: {val_acc:.4f}, F1: {val_f1:.4f}")
            print(f"  LR: {current_lr:.6f}")
            
            # Check for improvement
            is_best = val_f1 > self.best_val_f1
            if is_best:
                self.best_val_f1 = val_f1
                self.patience_counter = 0
            else:
                self.patience_counter += 1
            
            # Save checkpoint
            self.save_checkpoint(epoch, val_f1, is_best)
            self.save_history()
            
            # Early stopping
            if self.patience_counter >= self.patience:
                print(f"\nEarly stopping at epoch {epoch}")
                break
            
            print()
        
        total_time = time.time() - start_time
        print(f"\nTraining completed in {total_time/60:.1f} minutes")
        print(f"Best validation F1: {self.best_val_f1:.4f}")
        
        return self.history

In [ ]:
# Create fresh model for training
model = create_model(num_classes=6)
model = model.to(DEVICE)

# Create trainer
trainer = Trainer(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    device=DEVICE,
    lr=1e-3,
    weight_decay=1e-4,
    label_smoothing=0.1,
    epochs=50,
    patience=10
)

# Train!
history = trainer.train()

## Task 2.4 — Evaluation

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns


def evaluate_model(model, test_loader, device, class_names):
    """Full evaluation on test set."""
    model.eval()
    all_preds = []
    all_labels = []
    all_probs = []
    
    with torch.no_grad():
        for images, labels in tqdm(test_loader, desc="Evaluating"):
            images = images.to(device)
            outputs = model(images)
            probs = torch.softmax(outputs, dim=1)
            preds = outputs.argmax(dim=1)
            
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.numpy())
            all_probs.extend(probs.cpu().numpy())
    
    all_preds = np.array(all_preds)
    all_labels = np.array(all_labels)
    all_probs = np.array(all_probs)
    
    # Metrics
    accuracy = accuracy_score(all_labels, all_preds)
    f1 = f1_score(all_labels, all_preds, average="macro")
    
    print(f"\nTest Results:")
    print(f"  Top-1 Accuracy: {accuracy*100:.2f}%")
    print(f"  Macro F1: {f1:.4f}")
    
    # Classification report
    print("\nClassification Report:")
    print(classification_report(all_labels, all_preds, target_names=class_names))
    
    # Confusion matrix
    cm = confusion_matrix(all_labels, all_preds)
    plt.figure(figsize=(10, 8))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=class_names, yticklabels=class_names)
    plt.xlabel("Predicted")
    plt.ylabel("True")
    plt.title("Confusion Matrix")
    plt.tight_layout()
    plt.savefig(f"{RESULTS_DIR}/confusion_matrix.png", dpi=150)
    plt.show()
    
    return accuracy, f1, all_preds, all_labels, all_probs

In [ ]:
# Load best model
checkpoint = torch.load(f"{MODELS_DIR}/best_model.pth")
model.load_state_dict(checkpoint["model_state_dict"])
print(f"Loaded best model from epoch {checkpoint['epoch']} (Val F1: {checkpoint['val_f1']:.4f})")

# Evaluate on test set
accuracy, f1, preds, labels, probs = evaluate_model(model, test_loader, DEVICE, CLASS_NAMES)

## Task 2.5 — Grad-CAM

In [ ]:
import cv2
from typing import Optional


class GradCAM:
    """
    Grad-CAM for EfficientNet-B0.
    
    Hooks into the last convolutional layer (features.8) to compute
    class activation maps.
    """
    
    def __init__(self, model, target_layer=None):
        self.model = model
        self.model.eval()
        
        # Default: last conv block of EfficientNet-B0
        if target_layer is None:
            target_layer = model.features[-1]
        
        self.target_layer = target_layer
        self.gradients = None
        self.activations = None
        
        # Register hooks
        target_layer.register_forward_hook(self._forward_hook)
        target_layer.register_full_backward_hook(self._backward_hook)
    
    def _forward_hook(self, module, input, output):
        self.activations = output.detach()
    
    def _backward_hook(self, module, grad_input, grad_output):
        self.gradients = grad_output[0].detach()
    
    def generate(self, input_tensor, target_class: Optional[int] = None):
        """
        Generate Grad-CAM heatmap.
        
        Args:
            input_tensor: (1, 3, H, W) input image
            target_class: Class to visualize. If None, uses predicted class.
        
        Returns:
            heatmap: (H, W) normalized heatmap [0, 1]
            pred_class: Predicted class index
            confidence: Prediction confidence
        """
        self.model.zero_grad()
        
        # Forward pass
        output = self.model(input_tensor)
        probs = torch.softmax(output, dim=1)
        
        if target_class is None:
            target_class = output.argmax(dim=1).item()
        
        confidence = probs[0, target_class].item()
        
        # Backward pass for target class
        output[0, target_class].backward()
        
        # Compute weights (global average pooling of gradients)
        weights = self.gradients.mean(dim=(2, 3), keepdim=True)
        
        # Weighted combination of activation maps
        cam = (weights * self.activations).sum(dim=1, keepdim=True)
        
        # ReLU and normalize
        cam = torch.relu(cam)
        cam = cam.squeeze().cpu().numpy()
        
        # Normalize to [0, 1]
        cam = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)
        
        return cam, target_class, confidence
    
    def overlay_heatmap(self, image, heatmap, alpha=0.5):
        """
        Overlay heatmap on image.
        
        Args:
            image: (H, W) or (H, W, 3) original image [0, 255]
            heatmap: (h, w) heatmap from generate()
            alpha: Blend factor
        
        Returns:
            overlay: (H, W, 3) image with heatmap overlay
        """
        # Resize heatmap to image size
        h, w = image.shape[:2]
        heatmap_resized = cv2.resize(heatmap, (w, h))
        
        # Convert to colormap
        heatmap_colored = cv2.applyColorMap(
            (heatmap_resized * 255).astype(np.uint8),
            cv2.COLORMAP_JET
        )
        
        # Ensure image is 3-channel
        if len(image.shape) == 2:
            image = cv2.cvtColor(image, cv2.COLOR_GRAY2BGR)
        
        # Blend
        overlay = cv2.addWeighted(image, 1 - alpha, heatmap_colored, alpha, 0)
        
        return overlay

In [ ]:
# Generate Grad-CAM for 5 samples per class (30 total)
gradcam = GradCAM(model)

# Get test dataset for raw images
from src.dataset import NEUDataset
test_dataset = NEUDataset(DATA_DIR, split="test", preprocess=True)

# Sample 5 images per class
samples_per_class = {i: [] for i in range(6)}
for idx in range(len(test_dataset)):
    _, label = test_dataset.images[idx]
    if len(samples_per_class[label]) < 5:
        samples_per_class[label].append(idx)
    if all(len(v) >= 5 for v in samples_per_class.values()):
        break

# Generate and save Grad-CAMs
gradcam_dir = f"{RESULTS_DIR}/gradcam"
os.makedirs(gradcam_dir, exist_ok=True)

for class_idx, indices in samples_per_class.items():
    class_name = CLASS_NAMES[class_idx]
    print(f"\nGenerating Grad-CAM for {class_name}...")
    
    for i, idx in enumerate(indices):
        # Get image and tensor
        img_tensor, label = test_dataset[idx]
        img_path = test_dataset.images[idx][0]
        
        # Load original for overlay
        original = cv2.imread(str(img_path), cv2.IMREAD_GRAYSCALE)
        
        # Generate Grad-CAM
        input_tensor = img_tensor.unsqueeze(0).to(DEVICE)
        heatmap, pred_class, confidence = gradcam.generate(input_tensor)
        
        # Create overlay
        overlay = gradcam.overlay_heatmap(original, heatmap)
        
        # Save
        save_path = f"{gradcam_dir}/{class_name}_{i+1}.png"
        cv2.imwrite(save_path, overlay)
        
        pred_name = CLASS_NAMES[pred_class]
        print(f"  {i+1}. Pred: {pred_name} ({confidence:.2%})")

print(f"\nGrad-CAM images saved to {gradcam_dir}")

## Training Curves

In [ ]:
# Plot training curves
import pandas as pd

history_df = pd.DataFrame(history)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Loss
axes[0].plot(history_df["epoch"], history_df["train_loss"], label="Train")
axes[0].plot(history_df["epoch"], history_df["val_loss"], label="Val")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].set_title("Loss")
axes[0].legend()

# Accuracy
axes[1].plot(history_df["epoch"], history_df["train_acc"], label="Train")
axes[1].plot(history_df["epoch"], history_df["val_acc"], label="Val")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Accuracy")
axes[1].set_title("Accuracy")
axes[1].legend()

# F1
axes[2].plot(history_df["epoch"], history_df["train_f1"], label="Train")
axes[2].plot(history_df["epoch"], history_df["val_f1"], label="Val")
axes[2].set_xlabel("Epoch")
axes[2].set_ylabel("Macro F1")
axes[2].set_title("F1 Score")
axes[2].legend()

plt.tight_layout()
plt.savefig(f"{RESULTS_DIR}/training_curves.png", dpi=150)
plt.show()

## Summary

Fill in after running:

| Metric | Target | Actual |
|--------|--------|--------|
| Top-1 Accuracy | ≥97% | |
| Macro F1 (worst class) | ≥94% | |
| Training Time | ~20 min | |